In [1]:
!pip install -q sentencepiece transformers timm

In [2]:
!wget -q --show-progress "https://zenodo.org/record/2613548/files/cubicasa5k.zip" -O /content/cubicasa5k.zip
!unzip -q /content/cubicasa5k.zip -d /content/data/

/content/cubicasa5k 100%[===================>]   5.09G  25.4MB/s    in 3m 36s  


In [ ]:
import os
import re
import json
import math
import random
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image
from tqdm import tqdm
from lxml import etree

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T

import timm

from transformers import AutoTokenizer

4. Config

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

IMG_SIZE = 384
BATCH_SIZE = 8
EPOCHS = 30
MAX_LEN = 64
EMBED_DIM = 512
NUM_HEADS = 8
NUM_LAYERS = 6
DROPOUT = 0.1
LR = 2e-4

print('DEVICE:', DEVICE)

5. Semantic Room Mapping


In [ ]:
ROOM_MAP = {
    'LivingRoom':'living room',
    'Living':'living room',
    'Livingroom':'living room',
    'Dining':'dining room',
    'DiningRoom':'dining room',
    'Kitchen':'kitchen',
    'Bedroom':'bedroom',
    'MasterBedroom':'master bedroom',
    'Bathroom':'bathroom',
    'Toilet':'toilet',
    'WC':'toilet',
    'Hallway':'hallway',
    'Corridor':'hallway',
    'Entrance':'entrance',
    'Storage':'storage',
    'Closet':'closet',
    'WalkInCloset':'walk in closet',
    'Balcony':'balcony',
    'Terrace':'terrace',
    'Office':'office',
    'Garage':'garage',
    'Laundry':'laundry',
    'Room':'room',
}

6. SVG Parser

In [ ]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


def parse_svg(svg_path):
    try:
        tree = etree.parse(str(svg_path))
        root = tree.getroot()

        ns = re.match(r'\{(.+)\}', root.tag)
        ns = ns.group(1) if ns else 'http://www.w3.org/2000/svg'

        rooms = []
        seen = set()

        for elem in root.iter(f'{{{ns}}}g'):
            eid = (elem.get('id', '') + ' ' + elem.get('class', '')).lower()

            for eng, semantic in ROOM_MAP.items():
                if eng.lower() in eid and semantic not in seen:
                    seen.add(semantic)
                    rooms.append(semantic)

        for txt in root.iter(f'{{{ns}}}text'):
            t = ''.join(txt.itertext()).strip().lower()

            for eng, semantic in ROOM_MAP.items():
                if eng.lower() in t and semantic not in seen:
                    seen.add(semantic)
                    rooms.append(semantic)

        return rooms

    except:
        return []

7. Better Caption Generation

In [ ]:

def pluralize(room):
    irregular = {
        'balcony': 'balconies',
        'toilet': 'toilets',
        'bathroom': 'bathrooms',
        'bedroom': 'bedrooms',
        'kitchen': 'kitchens',
        'living room': 'living rooms',
    }
    return irregular.get(room, room + 's')

def make_semantic_caption(rooms):

    counts = Counter(rooms)

    parts = []

    for room, count in sorted(counts.items()):

        if count == 1:
            parts.append(f'1 {room}')
        else:
            parts.append(f'{count} {pluralize(room)}')

    caption = 'apartment floorplan containing ' + ', '.join(parts)  # Fix 4

    return caption

8. Build Dataset

In [ ]:
svg_files = list(Path('/content/data').rglob('*.svg'))

samples = []

for svg in tqdm(svg_files):

    folder = svg.parent

    images = list(folder.glob('*.png')) + list(folder.glob('*.jpg'))

    if len(images) == 0:
        continue

    rooms = parse_svg(svg)

    if len(rooms) == 0:
        continue

    caption = make_semantic_caption(rooms)

    samples.append({
        'image_path': str(images[0]),
        'caption': caption,
    })

print('DATASET SIZE:', len(samples))
print(samples[0])

9. Save Dataset

In [ ]:
with open('/content/data/floorplan_dataset.json', 'w') as f:
    json.dump(samples, f)

10. Better Tokenizer

In [ ]:
TOKENIZER_NAME = 'bert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

print(tokenizer.vocab_size)

11. Train / Validation Split


In [ ]:
random.shuffle(samples)

val_size = int(len(samples) * 0.1)

val_samples = samples[:val_size]
train_samples = samples[val_size:]

print(len(train_samples), len(val_samples))

12. Better Image Augmentation

In [ ]:
train_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomRotation(3),
    T.RandomHorizontalFlip(0.5),
    T.ColorJitter(0.1, 0.1, 0.1),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

13. Dataset Class

In [ ]:
class FloorplanDataset(Dataset):

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        sample = self.samples[idx]

        try:
            image = Image.open(sample['image_path']).convert('RGB')
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (255,255,255))

        image = self.transform(image)

        encoded = tokenizer(
            sample['caption'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LEN,
            return_tensors='pt'
        )

        input_ids = encoded['input_ids'].squeeze(0)
        attention_mask = encoded['attention_mask'].squeeze(0)

        return image, input_ids, attention_mask

14. Dataloaders

In [ ]:
train_loader = DataLoader(
    FloorplanDataset(train_samples, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    FloorplanDataset(val_samples, val_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)

15. Better Encoder

In [ ]:
# ── Spatial Encoder: ResNet-101 ──────────────────────────────────────────────
class SpatialEncoder(nn.Module):
    """ResNet-101 captures spatial structure (walls, room positions, layout)"""

    def __init__(self):
        super().__init__()

        resnet = timm.create_model('resnet101', pretrained=True, num_classes=0, global_pool='')

        # Freeze early layers, fine-tune layer3 and layer4
        for name, param in resnet.named_parameters():
            if 'layer3' in name or 'layer4' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.backbone = resnet
        self.proj = nn.Linear(2048, EMBED_DIM)  # ResNet-101 output: 2048

    def forward(self, x):
        feat = self.backbone.forward_features(x)   # (B, 2048, H, W)
        B, C, H, W = feat.shape
        feat = feat.flatten(2).transpose(1, 2)     # (B, H*W, 2048)
        feat = self.proj(feat)                      # (B, H*W, EMBED_DIM)
        return feat


# ── Semantic Encoder: EfficientNetV2-S ───────────────────────────────────────
class SemanticEncoder(nn.Module):
    """EfficientNetV2 captures semantic content (room types, textures)"""

    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            'tf_efficientnetv2_s',
            pretrained=True,
            num_classes=0,
            global_pool=''
        )

        # Freeze early layers, fine-tune blocks 4-6
        for name, param in self.backbone.named_parameters():
            if 'blocks.4' in name or 'blocks.5' in name or 'blocks.6' in name:
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.proj = nn.Linear(1280, EMBED_DIM)  # EfficientNetV2-S output: 1280

    def forward(self, x):
        feat = self.backbone.forward_features(x)   # (B, 1280, H, W)
        B, C, H, W = feat.shape
        feat = feat.flatten(2).transpose(1, 2)     # (B, H*W, 1280)
        feat = self.proj(feat)                      # (B, H*W, EMBED_DIM)
        return feat


# ── Cross-Attention Fusion ────────────────────────────────────────────────────
class CrossAttentionFusion(nn.Module):
    """Fuses spatial (ResNet) and semantic (EfficientNet) features via cross-attention"""

    def __init__(self, embed_dim=EMBED_DIM, num_heads=8):
        super().__init__()

        # Spatial features attend to semantic features
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, spatial, semantic):
        # spatial = ResNet features (query)
        # semantic = EfficientNet features (key, value)
        # Spatial features look at semantic features to enrich themselves
        fused, _ = self.cross_attn(query=spatial, key=semantic, value=semantic)
        fused = self.norm1(spatial + fused)         # residual connection
        fused = self.norm2(fused + self.ffn(fused)) # feed-forward + residual
        return fused


# ── Dual Encoder (replaces single Encoder) ───────────────────────────────────
class DualEncoder(nn.Module):
    """Combines ResNet-101 (spatial) + EfficientNetV2 (semantic) with cross-attention fusion"""

    def __init__(self):
        super().__init__()

        self.spatial_encoder  = SpatialEncoder()
        self.semantic_encoder = SemanticEncoder()
        self.fusion           = CrossAttentionFusion()

    def forward(self, x):
        spatial  = self.spatial_encoder(x)   # (B, N, EMBED_DIM)
        semantic = self.semantic_encoder(x)  # (B, M, EMBED_DIM)

        # Interpolate if spatial sizes differ
        if spatial.shape[1] != semantic.shape[1]:
            semantic = torch.nn.functional.interpolate(
                semantic.transpose(1, 2),
                size=spatial.shape[1],
                mode='linear',
                align_corners=False
            ).transpose(1, 2)

        fused = self.fusion(spatial, semantic)  # (B, N, EMBED_DIM)
        return fused


# Keep Encoder as alias for backward compatibility
Encoder = DualEncoder

16. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

17. Transformer Decoder

In [ ]:
class CaptionTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            tokenizer.vocab_size,
            EMBED_DIM
        )

        self.positional = PositionalEncoding(EMBED_DIM)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=EMBED_DIM,
            nhead=NUM_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=NUM_LAYERS
        )

        self.fc = nn.Linear(EMBED_DIM, tokenizer.vocab_size)

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, memory, tokens):

        x = self.embedding(tokens)

        x = self.positional(x)

        x = self.dropout(x)  # Fix 3: dropout after embeddings

        seq_len = tokens.shape[1]

        mask = torch.triu(
            torch.ones(seq_len, seq_len),
            diagonal=1
        ).bool().to(tokens.device)

        padding_mask = (tokens == tokenizer.pad_token_id)  # Fix 2: padding mask

        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=mask,
            tgt_key_padding_mask=padding_mask  # Fix 2
        )

        out = self.fc(out)

        return out

18. Full Model

In [ ]:
class FloorplanCaptioningModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = Encoder()
        self.decoder = CaptionTransformer()

    def forward(self, images, tokens):

        memory = self.encoder(images)

        out = self.decoder(memory, tokens)

        return out

19. Initialize Model

In [ ]:
model = FloorplanCaptioningModel().to(DEVICE)

print(sum(p.numel() for p in model.parameters() if p.requires_grad))

20. Optimizer + Loss

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

21. Early Stopping

In [ ]:
class EarlyStopping:

    def __init__(self, patience=5):
        self.patience = patience
        self.best = 1e9
        self.counter = 0

    def step(self, value):

        if value < self.best:
            self.best = value
            self.counter = 0
            return False

        self.counter += 1

        return self.counter >= self.patience

early_stopping = EarlyStopping(patience=5)

22. Training Loop

In [ ]:
best_val = 1e9

for epoch in range(EPOCHS):

    # =========================
    # TRAIN
    # =========================
    model.train()

    train_loss = 0

    for images, input_ids, attention_mask in tqdm(train_loader):

        images = images.to(DEVICE)
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        optimizer.zero_grad()

        # Teacher forcing
        outputs = model(
            images,
            input_ids[:, :-1]
        )

        targets = input_ids[:, 1:]

        loss = criterion(
            outputs.reshape(-1, tokenizer.vocab_size),
            targets.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0

    with torch.no_grad():

        for images, input_ids, attention_mask in val_loader:

            images = images.to(DEVICE)
            input_ids = input_ids.to(DEVICE)
            attention_mask = attention_mask.to(DEVICE)

            outputs = model(
                images,
                input_ids[:, :-1]
            )

            targets = input_ids[:, 1:]

            loss = criterion(
                outputs.reshape(-1, tokenizer.vocab_size),
                targets.reshape(-1)
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    scheduler.step()

    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print(f'Train Loss: {train_loss:.4f}')
    print(f'Val Loss:   {val_loss:.4f}')

    # =========================
    # SAVE BEST MODEL
    # =========================
    if val_loss < best_val:

        best_val = val_loss

        torch.save(
            model.state_dict(),
            '/content/best_floorplan_model.pth'
        )

        print('✅ BEST MODEL SAVED')

    # =========================
    # EARLY STOPPING
    # =========================
    if early_stopping.step(val_loss):

        print('⛔ EARLY STOPPING')

        break

1. Load Best Model

In [ ]:
model.load_state_dict(
    torch.load('/content/best_floorplan_model.pth')
)

model.eval()

print("✅ Best model loaded")

2. Beam Search Caption Generation

In [ ]:
@torch.no_grad()
def generate_caption(model, image_path, beam_size=3):

    model.eval()

    image = Image.open(image_path).convert('RGB')

    image = val_transform(image)

    image = image.unsqueeze(0).to(DEVICE)

    memory = model.encoder(image)

    start_token = tokenizer.cls_token_id
    end_token = tokenizer.sep_token_id

    sequences = [([start_token], 0.0)]

    for _ in range(MAX_LEN):

        all_candidates = []

        for seq, score in sequences:

            if seq[-1] == end_token:
                all_candidates.append((seq, score))
                continue

            tokens = torch.tensor(seq).unsqueeze(0).to(DEVICE)

            outputs = model.decoder(memory, tokens)

            logits = outputs[:, -1, :]

            probs = F.log_softmax(logits, dim=-1)

            topk_probs, topk_ids = torch.topk(
                probs,
                beam_size
            )

            for k in range(beam_size):

                candidate = (
                    seq + [topk_ids[0][k].item()],
                    score + topk_probs[0][k].item()
                )

                all_candidates.append(candidate)

        ordered = sorted(
            all_candidates,
            key=lambda x: x[1] / max(len(x[0]), 1),  # length normalization
            reverse=True
        )

        sequences = ordered[:beam_size]

    best_sequence = sequences[0][0]

    caption = tokenizer.decode(
        best_sequence,
        skip_special_tokens=True
    )

    return caption

3. Test on Validation Samples

In [ ]:
for i in range(5):

    sample = random.choice(val_samples)

    print("=" * 80)

    print("IMAGE:")
    print(sample['image_path'])

    print("\nGROUND TRUTH:")
    print(sample['caption'])

    pred = generate_caption(
        model,
        sample['image_path']
    )

    print("\nPREDICTION:")
    print(pred)

    print()

4. Visualize Predictions

In [ ]:
import matplotlib.pyplot as plt

sample = random.choice(val_samples)

image = Image.open(sample['image_path']).convert('RGB')

pred = generate_caption(
    model,
    sample['image_path']
)

plt.figure(figsize=(10,10))
plt.imshow(image)
plt.axis('off')

plt.title(pred)

plt.show()